<a href="https://colab.research.google.com/github/goumze/Simplilearn_Agentic_AI/blob/feature%2Fcollab/Simple_LangChain_vs_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LangGraph Application with OpenAI `gpt-4o-mini`

This notebook demonstrates a very basic LangGraph application that integrates with OpenAI's `gpt-4o-mini` model. LangGraph is a library for building stateful, multi-actor applications with LLMs, ideal for creating conversational agents, planning systems, and more complex workflows.

### 1. Install Necessary Libraries

First, we need to install the `langchain-openai` package to connect to OpenAI models and `langgraph` for building the graph structure.

In [ ]:
# Install libraries
!pip install -qq langchain-openai langgraph

### 2. Set Up OpenAI API Key

To use OpenAI models, you need an API key. Please ensure you have your OpenAI API key stored in Colab secrets under the name `OPENAI_API_KEY`.

In [ ]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI
from typing import List, Tuple, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage

# Retrieve API key from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# Initialize the OpenAI Chat Model
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7)
print(f"Initialized LLM: {llm.model_name}")

Initialized LLM: gpt-4o-mini


### 3. Define the Graph State

LangGraph operates on a "state" object that is passed between nodes. For a simple conversational agent, the state can be a list of messages.

In [ ]:
class AgentState(TypedDict):
    messages: List[BaseMessage]

print("Defined AgentState with a list of messages.")

Defined AgentState with a list of messages.


### 4. Define the Agent Node

The agent node is a function that takes the current state, processes it (e.g., calls the LLM), and returns an updated state.

In [ ]:
def call_model(state: AgentState) -> AgentState:
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages': messages + [response]}

print("Defined 'call_model' function as the agent node.")

Defined 'call_model' function as the agent node.


### 5. Build and Compile the LangGraph

Now, we'll use `StateGraph` to define the flow. In this simple case, we have only one node (`call_model`) and the graph will simply call this node.

In [ ]:
from langgraph.graph import StateGraph, START

# Create a StateGraph
workflow = StateGraph(AgentState)

# Add the agent node
workflow.add_node("llm_agent", call_model)

# Set the entry point to our llm_agent node
workflow.set_entry_point("llm_agent")

# Set the exit point to our llm_agent node (for a single step process)
workflow.set_finish_point("llm_agent")

# Compile the graph
app = workflow.compile()

print("LangGraph workflow built and compiled.")

LangGraph workflow built and compiled.


### 6. Run the LangGraph Application

Finally, let's invoke our compiled graph with an initial message and see the response from the `gpt-4o-mini` model.

In [ ]:
# Example usage
initial_message = "Tell me a short, interesting fact about space."

print(f"\nHuman: {initial_message}")

# Invoke the graph with the initial state
output = app.invoke({"messages": [HumanMessage(content=initial_message)]})

# Extract and print the AI's response
ai_response = output['messages'][-1].content
print(f"AI: {ai_response}")

print("\n--- Another example ---")
initial_message_2 = "Explain the concept of quantum entanglement in simple terms."
print(f"\nHuman: {initial_message_2}")
output_2 = app.invoke({"messages": [HumanMessage(content=initial_message_2)]})
ai_response_2 = output_2['messages'][-1].content
print(f"AI: {ai_response_2}")


Human: Tell me a short, interesting fact about space.
AI: One fascinating fact about space is that it is completely silent. Unlike on Earth, where sound travels through air or water, space is a vacuum, meaning there are no molecules to carry sound waves. So if an event like a supernova explosion occurs in space, it can be visually spectacular but completely silent!

--- Another example ---

Human: Explain the concept of quantum entanglement in simple terms.
AI: Quantum entanglement is a phenomenon in quantum physics where two or more particles become linked or connected in such a way that the state of one particle instantly influences the state of the other, no matter how far apart they are. 

Imagine you have a pair of magic dice. When you roll one die, the outcome of the second die is determined instantly, even if it's on the other side of the universe. If you roll a six on the first die, the second die will instantly show a three, for example, regardless of the distance between the

### 7. Visualize the LangGraph Structure

LangGraph allows you to visualize the structure of your compiled graph using Mermaid. This helps in understanding the flow and connections between nodes.

In [ ]:
# Generate the Mermaid diagram string
mermaid_graph = app.get_graph().draw_mermaid()

print("Generated Mermaid graph string.")

Generated Mermaid graph string.


```mermaid
{{mermaid_graph}}
```

*Note: If the Mermaid diagram doesn't render directly in your environment, you can copy the generated string from the output of the previous cell and paste it into an online Mermaid editor to visualize it.*

### 8. Advanced LangGraph Application: Agent with Tool Use

This example demonstrates a more sophisticated LangGraph application where the agent can decide to use a tool based on the user's input, showcasing conditional routing and tool invocation. This kind of dynamic decision-making is where LangGraph truly shines compared to a simple sequential LangChain chain.

#### 8.1 Define a Tool

First, let's define a simple dummy tool that our agent can potentially use. For a real application, this could be a search engine, a calculator, a database query, or any external API.

In [ ]:
from langchain_core.tools import tool

@tool
def search(query: str) -> str:
    """Searches the web for the given query and returns relevant information."""
    # In a real application, this would call an actual search API (e.g., Google Search, Tavily)
    # For this example, we'll return a static response.
    if "weather" in query.lower():
        return "The current weather in London is 15°C and partly cloudy."
    elif "capital of france" in query.lower():
        return "The capital of France is Paris."
    else:
        return f"No specific search result for '{query}'. This is a dummy search tool."

tools = [search]
print("Defined 'search' tool.")

Defined 'search' tool.


#### 8.2 Initialize LLM with Tool-Calling Capabilities

We need to initialize a new instance of `ChatOpenAI` and bind our `search` tool to it. This tells the LLM that it has this tool available and can generate `tool_calls` in its response.

In [ ]:
llm_with_tools = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7).bind_tools(tools)
print(f"Initialized LLM with tools: {llm_with_tools.model_name}")

Initialized LLM with tools: gpt-4o-mini


#### 8.3 Define Nodes for the Advanced Graph

We'll need two main types of nodes:

1.  **LLM Agent Node:** This node will invoke our `llm_with_tools`.
2.  **Tool Node:** This node will execute any tool calls suggested by the LLM.

In [ ]:
# Node to invoke the LLM with tools
def call_llm_with_tools(state: AgentState) -> AgentState:
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': messages + [response]}

# Node to execute tool calls
def call_tool_node(state: AgentState) -> AgentState:
    last_message = state['messages'][-1]
    tool_outputs = []
    for tool_call in last_message.tool_calls:
        if tool_call.get('name') == 'search':
            output = search.invoke(tool_call['args'])
            tool_outputs.append(ToolMessage(content=output, tool_call_id=tool_call['id']))
        # Add more tool handling here if you have multiple tools
    return {'messages': state['messages'] + tool_outputs}

print("Defined 'call_llm_with_tools' and 'call_tool_node' functions.")

Defined 'call_llm_with_tools' and 'call_tool_node' functions.


#### 8.4 Define the Routing Logic

This function will act as our router, determining the next step in the graph based on the LLM's response. If the LLM suggested a tool call, we go to the tool node; otherwise, the conversation continues or ends.

In [ ]:
def decide_next_step(state: AgentState) -> str:
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        return "call_tool"
    else:
        return "__end__"

print("Defined 'decide_next_step' routing function.")

Defined 'decide_next_step' routing function.


#### 8.5 Build and Compile the Advanced LangGraph

Now we'll put it all together with conditional edges, allowing the graph to dynamically switch between calling the LLM, using a tool, and ending the process.

In [ ]:
from langgraph.graph import StateGraph, END

# Create a new StateGraph for the advanced example
workflow_advanced = StateGraph(AgentState)

# Add the nodes
workflow_advanced.add_node("llm_agent", call_llm_with_tools)
workflow_advanced.add_node("tool_node", call_tool_node)

# Set the entry point
workflow_advanced.set_entry_point("llm_agent")

# Add conditional edges
# From llm_agent, decide if we need to call a tool or end
workflow_advanced.add_conditional_edges(
    "llm_agent",
    decide_next_step,
    {
        "call_tool": "tool_node",
        "__end__": END
    }
)

# After calling a tool, send the output back to the LLM for processing
workflow_advanced.add_edge('tool_node', 'llm_agent')

# Compile the advanced graph
app_advanced = workflow_advanced.compile()

print("Advanced LangGraph workflow built and compiled.")

Advanced LangGraph workflow built and compiled.


#### 8.6 Run the Advanced LangGraph Application

Let's test our advanced agent with tool-calling capabilities. You'll see how it intelligently decides to use the `search` tool when needed.

In [ ]:
# Example 1: Query requiring a tool call
initial_message_tool = "What's the weather in London?"
print(f"\nHuman: {initial_message_tool}")

output_tool = app_advanced.invoke({"messages": [HumanMessage(content=initial_message_tool)]})

ai_response_tool = output_tool['messages'][-1].content
print(f"AI: {ai_response_tool}")

print("\n---")

# Example 2: Query that the LLM can answer directly
initial_message_direct = "Tell me a funny joke."
print(f"\nHuman: {initial_message_direct}")

output_direct = app_advanced.invoke({"messages": [HumanMessage(content=initial_message_direct)]})

ai_response_direct = output_direct['messages'][-1].content
print(f"AI: {ai_response_direct}")

print("\n---")

# Example 3: Another query requiring a tool call (that should get a dummy response)
initial_message_tool_2 = "Search for the latest news on AI development."
print(f"\nHuman: {initial_message_tool_2}")

output_tool_2 = app_advanced.invoke({"messages": [HumanMessage(content=initial_message_tool_2)]})

ai_response_tool_2 = output_tool_2['messages'][-1].content
print(f"AI: {ai_response_tool_2}")


Human: What's the weather in London?
AI: The current weather in London is 15°C and partly cloudy.

---

Human: Tell me a funny joke.
AI: Why don't scientists trust atoms?

Because they make up everything!

---

Human: Search for the latest news on AI development.
AI: I wasn't able to retrieve the latest news on AI development at the moment. However, you can check reputable news websites, technology blogs, or specific AI research journals for the most recent updates. If there's anything specific you want to know about AI, feel free to ask!


#### 8.7 Visualize the Advanced LangGraph Structure

Let's visualize the more complex structure of this advanced graph, showing the conditional edges.

In [ ]:
# Generate the Mermaid diagram string for the advanced graph
mermaid_graph_advanced = app_advanced.get_graph().draw_mermaid()

print("Generated Mermaid graph string for advanced example.")

Generated Mermaid graph string for advanced example.


```mermaid
{{mermaid_graph_advanced}}
```

*Note: This diagram illustrates how the `llm_agent` can conditionally transition to either the `tool_node` or the `END` state, and how `tool_node` feeds its results back to `llm_agent` for further processing, creating a powerful multi-step, dynamic workflow.*